In [1]:
import sys

# Check if we are currently running inside Google Colab
if 'google.colab' in sys.modules:
    print("Colab environment detected. Installing dependencies and cloning repository...")
    # Major project dependencies used by this notebook and local modules.
    colab_requirements = [
        'numpy',
        'sentencepiece',
        'datasets',
        'jax[cpu]',
        'matplotlib',
        'ipykernel',
    ]

    !pip install -q {" ".join(colab_requirements)}
    
    # Run the shell command to clone
    !git clone https://github.com/krishnan1159/NMT.git
    
    # Add the cloned repo to the system path
    sys.path.append('/content/NMT')
    
else:
    print("Local environment detected. Skipping git clone.")

Colab environment detected. Installing dependencies and cloning repository...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.1 MB/s eta 0:00:00a 0:00:01
Cloning into 'NMT'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 2), reused 17 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 343.18 KiB | 3.36 MiB/s, done.
Resolving deltas: 100% (2/2), done.


In [3]:
#%load_ext autoreload
#%autoreload 2
#%reload_ext autoreload

## Local classes
import importlib
from vocab import Tokenizer
from model_embedding import ModelEmbedding
from config import get_config
import rnn

importlib.reload(rnn)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


<module 'rnn' from '/content/NMT/rnn.py'>

In [4]:
from datasets import load_dataset
import jax
import jax.numpy as jp

In [5]:
ds = load_dataset("shiyue/chr_en", "parallel")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/7.55k [00:00<?, ?B/s]

parallel/train-00000-of-00001.parquet:   0%|          | 0.00/1.75M [00:00<?, ?B/s]

parallel/dev-00000-of-00001.parquet:   0%|          | 0.00/149k [00:00<?, ?B/s]

parallel/out_dev-00000-of-00001.parquet:   0%|          | 0.00/46.3k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/11639 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating out_dev split:   0%|          | 0/256 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating out_test split:   0%|          | 0/256 [00:00<?, ? examples/s]

In [6]:
train_set = list(ds["train"]["sentence_pair"])
chr_sentences = [sentence['chr'] for sentence in train_set]
eng_sentences = [sentence['en'] for sentence in train_set]

## Generate vocab for both source and target sentence.
cfg = get_config()
chr_vocab = Tokenizer(chr_sentences, "chr", "chr", "20000")
eng_vocab = Tokenizer(eng_sentences, "eng", "eng", "8000")
random_key = jax.random.key(0)
embedding = ModelEmbedding(chr_vocab.get_vocab_size(), eng_vocab.get_vocab_size(), cfg.embed_size, random_key)

In [7]:
print(eng_sentences[0])
a, b = eng_vocab.to_numpy([eng_sentences[1]])
print(b)
print(a.shape)
embedding.target_embed(a).shape

and by him every one that believeth is justified from all things, from which ye could not be justified by the law of Moses.
[29]
(1, 29)


(1, 29, 128)

In [9]:
#chr_sentences = chr_sentences[:140]
#eng_sentences = eng_sentences[:140]

chr_tokens, chr_lengths = chr_vocab.to_numpy(chr_sentences, add_bos=True, add_eos=True)
eng_tokens, eng_lengths = eng_vocab.to_numpy(eng_sentences, add_bos=True, add_eos=True)

rnn.train(chr_tokens, chr_lengths, eng_tokens, eng_lengths, embedding, eng_vocab.get_vocab_size(), cfg)

INFO:rnn:JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
INFO:rnn:JAX backend: tpu
INFO:rnn:RNN cumulative params : embedding = 3584000, encoder = 32769, decoder = 1056769 and total = 4673538
INFO:rnn:Loss at epoch 1/10 is 8.987151145935059. Time taken to train = 0.005406500000049164
INFO:rnn:Loss at epoch 2/10 is 8.986983299255371. Time taken to train = 0.005418519000045308
INFO:rnn:Loss at epoch 3/10 is 8.986673355102539. Time taken to train = 0.005422149999958492
INFO:rnn:Loss at epoch 4/10 is 8.985990524291992. Time taken to train = 0.00533759999996164
INFO:rnn:Loss at epoch 5/10 is 8.984444618225098. Time taken to train = 0.005369790000031571
INFO:rnn:Loss at epoch 6/10 is 8.980935096740723. Time taken to train = 0.0053151089999801115
INFO:rnn:Loss at epoch 7/10 is 8.972986221313477. Time taken to train = 0.005331589999968855
INFO:rnn:Loss at epoch 8/10 is 8.95494556427002. Time taken to train = 0.005211249000012685
INFO:rnn:Loss at epoch 9/10 is 8.